# 📦 Task 1: Data Collection
---
**Objective:** Gather data from a public API and a reliable dataset source, then save it in a structured format (CSV & JSON) for downstream analysis.

**Dataset Used:** [World Bank Open Data API](https://api.worldbank.org/) — GDP per Capita (current US$) for all countries (2000–2022).

**API Endpoint:**
```
https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=1000&date=2000:2022
```

---
**Author:** Data Science Lab  
**Tools:** Python, Pandas, Requests, JSON


## 🔧 Step 1: Install & Import Required Libraries

In [ ]:
# Install required libraries (if not already installed)
!pip install requests pandas tabulate --quiet

# Core libraries
import requests
import pandas as pd
import json
import os
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"📅 Run Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


✅ All libraries imported successfully!
📅 Run Date: 2026-05-30 07:13:29


## ⚙️ Step 2: API Configuration & Parameters

In [ ]:
# ── API Configuration ──────────────────────────────────────────────────
BASE_URL    = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD"
PARAMS      = {
    "format"   : "json",
    "per_page" : 1000,
    "date"     : "2000:2022",
}

OUTPUT_DIR  = "collected_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("🌐 API Base URL :", BASE_URL)
print("📋 Parameters   :", json.dumps(PARAMS, indent=2))
print(f"📁 Output folder: {OUTPUT_DIR}/")


🌐 API Base URL : https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD
📋 Parameters   : {
  "format": "json",
  "per_page": 1000,
  "date": "2000:2022"
}
📁 Output folder: collected_data/


## 🌐 Step 3: Fetch Data from World Bank API (Paginated)

In [ ]:
def fetch_all_pages(base_url: str, params: dict) -> list:
    """Fetch all paginated results from the World Bank API."""
    all_records = []
    page        = 1
    total_pages = None

    print("⏳ Fetching data from World Bank API...")
    while True:
        params["page"] = page
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()

        payload = response.json()
        meta, records = payload[0], payload[1]

        if total_pages is None:
            total_pages = meta["pages"]
            total_records = meta["total"]
            print(f"   📊 Total records : {total_records:,}")
            print(f"   📄 Total pages   : {total_pages}")

        if records:
            all_records.extend(records)

        print(f"   ✔  Page {page}/{total_pages} fetched  ({len(records)} records)")
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.2)           # polite delay

    print(f"\n✅ Fetched {len(all_records):,} raw records in total.")
    return all_records

raw_records = fetch_all_pages(BASE_URL, PARAMS.copy())


⏳ Fetching data from World Bank API...
   📊 Total records : 6,118
   📄 Total pages   : 7
   ✔  Page 1/7 fetched  (1000 records)
   ✔  Page 2/7 fetched  (1000 records)
   ✔  Page 3/7 fetched  (1000 records)
   ✔  Page 4/7 fetched  (1000 records)
   ✔  Page 5/7 fetched  (1000 records)
   ✔  Page 6/7 fetched  (1000 records)
   ✔  Page 7/7 fetched  (118 records)

✅ Fetched 6,118 raw records in total.


## 🧹 Step 4: Parse Raw JSON into a Structured DataFrame

In [ ]:
def parse_records(records: list) -> pd.DataFrame:
    """Flatten nested World Bank JSON into a tidy DataFrame."""
    rows = []
    for rec in records:
        rows.append({
            "country_id"    : rec["country"]["id"],
            "country_name"  : rec["country"]["value"],
            "indicator_id"  : rec["indicator"]["id"],
            "indicator_name": rec["indicator"]["value"],
            "year"          : int(rec["date"]),
            "gdp_per_capita": rec["value"],    # USD, can be None
        })
    return pd.DataFrame(rows)

df_raw = parse_records(raw_records)

print("📐 Raw DataFrame shape :", df_raw.shape)
print("\n🔍 Data Types:")
print(df_raw.dtypes)
print("\n📋 First 5 rows:")
df_raw.head()


📐 Raw DataFrame shape : (6118, 6)

🔍 Data Types:
country_id         object
country_name       object
indicator_id       object
indicator_name     object
year                int64
gdp_per_capita    float64
dtype: object

📋 First 5 rows:


,country_id,country_name,indicator_id,indicator_name,year,gdp_per_capita
0,ZH,Africa Eastern and Southern,NY.GDP.PCAP.CD,GDP per capita (current US$),2022,1679.327622
1,ZH,Africa Eastern and Southern,NY.GDP.PCAP.CD,GDP per capita (current US$),2021,1562.416175
2,ZH,Africa Eastern and Southern,NY.GDP.PCAP.CD,GDP per capita (current US$),2020,1351.591669
3,ZH,Africa Eastern and Southern,NY.GDP.PCAP.CD,GDP per capita (current US$),2019,1507.085600
4,ZH,Africa Eastern and Southern,NY.GDP.PCAP.CD,GDP per capita (current US$),2018,1552.073722


## 📊 Step 5: Data Quality Overview

In [ ]:
print("=" * 55)
print("        DATA QUALITY SUMMARY — RAW DATASET")
print("=" * 55)
print(f"  Rows                : {df_raw.shape[0]:>10,}")
print(f"  Columns             : {df_raw.shape[1]:>10}")
print(f"  Unique Countries    : {df_raw['country_name'].nunique():>10,}")
print(f"  Year Range          : {df_raw['year'].min()} – {df_raw['year'].max()}")
print(f"  Missing GDP values  : {df_raw['gdp_per_capita'].isna().sum():>10,}")
print(f"  Completeness        : {(1 - df_raw['gdp_per_capita'].isna().mean())*100:>9.1f}%")
print("=" * 55)

print("\n📊 Descriptive Statistics (numeric):")
df_raw.describe()


        DATA QUALITY SUMMARY — RAW DATASET
  Rows                :      6,118
  Columns             :          6
  Unique Countries    :        266
  Year Range          : 2000 – 2022
  Missing GDP values  :        176
  Completeness        :      97.1%

📊 Descriptive Statistics (numeric):


,year,gdp_per_capita
count,6118.000000,5942.000000
mean,2011.000000,14492.633146
std,6.633792,22869.979630
min,2000.000000,109.593814
25%,2005.000000,1525.201231
50%,2011.000000,4911.768565
75%,2017.000000,18205.557910
max,2022.000000,226052.001905


## 🎯 Step 6: Filter & Enrich the Dataset

In [ ]:
# Keep only rows with a reported GDP value
df_clean = df_raw.dropna(subset=["gdp_per_capita"]).copy()

# Add a human-readable income band
def income_band(val):
    if val < 1_000:   return "Low Income"
    if val < 4_000:   return "Lower-Middle Income"
    if val < 13_000:  return "Upper-Middle Income"
    return "High Income"

df_clean["income_band"] = df_clean["gdp_per_capita"].apply(income_band)

# Add collection metadata
df_clean["collected_at"] = datetime.now().strftime("%Y-%m-%d")
df_clean["data_source"]  = "World Bank API — NY.GDP.PCAP.CD"

print(f"📐 Cleaned DataFrame shape : {df_clean.shape}")
print("\n📊 Income Band Distribution:")
print(df_clean["income_band"].value_counts().to_string())
print("\n🔍 Sample rows:")
df_clean.sample(5, random_state=42)


📐 Cleaned DataFrame shape : (5942, 9)

📊 Income Band Distribution:
income_band
High Income            1773
Lower-Middle Income    1675
Upper-Middle Income    1499
Low Income              995

🔍 Sample rows:


,country_id,country_name,indicator_id,indicator_name,year,gdp_per_capita,income_band,collected_at,data_source
4892,WS,Samoa,NY.GDP.PCAP.CD,GDP per capita (current US$),2006,2663.291359,Lower-Middle Income,2026-05-30,World Bank API — NY.GDP.PCAP.CD
2186,CR,Costa Rica,NY.GDP.PCAP.CD,GDP per capita (current US$),2021,12838.118536,Upper-Middle Income,2026-05-30,World Bank API — NY.GDP.PCAP.CD
1425,AZ,Azerbaijan,NY.GDP.PCAP.CD,GDP per capita (current US$),2000,655.097250,Low Income,2026-05-30,World Bank API — NY.GDP.PCAP.CD
5333,LC,St. Lucia,NY.GDP.PCAP.CD,GDP per capita (current US$),2002,5567.894284,Upper-Middle Income,2026-05-30,World Bank API — NY.GDP.PCAP.CD
2918,GL,Greenland,NY.GDP.PCAP.CD,GDP per capita (current US$),2002,20657.950632,High Income,2026-05-30,World Bank API — NY.GDP.PCAP.CD


## 💾 Step 7: Save Dataset as CSV

In [ ]:
csv_path = os.path.join(OUTPUT_DIR, "world_bank_gdp_per_capita.csv")
df_clean.to_csv(csv_path, index=False)
print(f"✅ CSV saved  → {csv_path}")
print(f"   Size : {os.path.getsize(csv_path)/1024:.1f} KB  |  Rows : {len(df_clean):,}")


✅ CSV saved  → collected_data/world_bank_gdp_per_capita.csv
   Size : 833.7 KB  |  Rows : 5,942


## 💾 Step 8: Save Dataset as JSON

In [ ]:
json_path = os.path.join(OUTPUT_DIR, "world_bank_gdp_per_capita.json")
df_clean.to_json(json_path, orient="records", indent=2)
print(f"✅ JSON saved → {json_path}")
print(f"   Size : {os.path.getsize(json_path)/1024:.1f} KB  |  Records : {len(df_clean):,}")

# Preview the JSON structure
with open(json_path) as f:
    sample = json.load(f)
print("\n🔍 JSON structure preview (first record):")
print(json.dumps(sample[0], indent=4))


✅ JSON saved → collected_data/world_bank_gdp_per_capita.json
   Size : 1971.3 KB  |  Records : 5,942

🔍 JSON structure preview (first record):
{
    "country_id": "ZH",
    "country_name": "Africa Eastern and Southern",
    "indicator_id": "NY.GDP.PCAP.CD",
    "indicator_name": "GDP per capita (current US$)",
    "year": 2022,
    "gdp_per_capita": 1679.3276216647,
    "income_band": "Lower-Middle Income",
    "collected_at": "2026-05-30",
    "data_source": "World Bank API \u2014 NY.GDP.PCAP.CD"
}


## 📋 Step 9: Save Collection Metadata Manifest

In [ ]:
manifest = {
    "collection_timestamp": datetime.now().isoformat(),
    "source"              : "World Bank Open Data API",
    "indicator"           : "NY.GDP.PCAP.CD — GDP per Capita (current US$)",
    "date_range"          : f"{df_clean['year'].min()} – {df_clean['year'].max()}",
    "countries_collected" : int(df_clean["country_name"].nunique()),
    "total_records"       : len(df_clean),
    "missing_values"      : int(df_raw["gdp_per_capita"].isna().sum()),
    "output_files"        : {
        "csv"  : csv_path,
        "json" : json_path,
    },
    "columns": list(df_clean.columns),
}

manifest_path = os.path.join(OUTPUT_DIR, "collection_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("✅ Manifest saved →", manifest_path)
print("\n📋 Manifest Contents:")
print(json.dumps(manifest, indent=4))


✅ Manifest saved → collected_data/collection_manifest.json

📋 Manifest Contents:
{
    "collection_timestamp": "2026-05-30T07:14:27.324879",
    "source": "World Bank Open Data API",
    "indicator": "NY.GDP.PCAP.CD \u2014 GDP per Capita (current US$)",
    "date_range": "2000 \u2013 2022",
    "countries_collected": 262,
    "total_records": 5942,
    "missing_values": 176,
    "output_files": {
        "csv": "collected_data/world_bank_gdp_per_capita.csv",
        "json": "collected_data/world_bank_gdp_per_capita.json"
    },
    "columns": [
        "country_id",
        "country_name",
        "indicator_id",
        "indicator_name",
        "year",
        "gdp_per_capita",
        "income_band",
        "collected_at",
        "data_source"
    ]
}


## ✅ Step 10: Verify Saved Files

In [ ]:
print("📁 Files in output directory:\n")
for fname in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f"  📄 {fname:<45}  {os.path.getsize(fpath)/1024:>8.1f} KB")

# Reload CSV to confirm integrity
df_verify = pd.read_csv(csv_path)
print(f"\n🔄 CSV reloaded successfully — Shape: {df_verify.shape}")
assert df_verify.shape == df_clean.shape, "❌ Shape mismatch!"
print("✅ Data integrity check PASSED")


📁 Files in output directory:

  📄 world_bank_gdp_per_capita.csv                     833.7 KB
  📄 world_bank_gdp_per_capita.json                   1971.3 KB
  📄 collection_manifest.json                            0.7 KB

🔄 CSV reloaded successfully — Shape: (5942, 9)
✅ Data integrity check PASSED


---
## 📝 Task 1 Summary

| Item | Detail |
|------|--------|
| **Data Source** | World Bank Open Data API |
| **Indicator** | GDP per Capita (NY.GDP.PCAP.CD) |
| **Years Covered** | 2000 – 2022 |
| **Countries** | 200+ |
| **Records Collected** | 4,000+ |
| **Formats Saved** | CSV, JSON, Metadata Manifest |
| **Key Feature Added** | Income Band classification |

> ✅ **Task 1 Complete** — Dataset collected, parsed, enriched, and persisted in both CSV and JSON formats with a collection manifest for reproducibility.
